# Selection

**Choosing the most relevant features** from the original set to reduce noise, improve model performance, and lower computational cost.
- We must use features with a strong predictive signal for our task
- Supported by intuition and domain knowledge

"In practice, mastering feature engineering involves experimentation and collaboration with domain experts."

**Important:** More features ≠ better model
> - Irrelevant or redundant (covariant) features add noise, increase overfitting risk, and slow training
> - Unused features add technical debt
> - Better feature selection will reduce error rate

## Key Methods

- Domain Knowledge/Experimentation
- Filter Methods
- Wrapper Methods
- Embedded Methods

| Method | Best For | Drawbacks |
|:-------|:---------|:----------|
| Filter | Fast preprocessing, high-dimensional data | Ignores feature interactions |
| Wrapper | Optimal subset for specific model | Computationally expensive, overfitting risk |
| Embedded | Balance of speed and performance | Tied to specific model type |

In [64]:
# Import necessary example packages
import numpy as np
import pandas as pd

## Filter Methods
Evaluate features independently using statistical measures before modeling.
>- **Variance threshold**: Remove low-variance features (near-constant values)
>- **Correlation threshold**: Drop features highly correlated with each other
>- **Chi-square test**: Measure dependence between categorical features and target
>- **Mutual information**: Capture non-linear relationships with target

### Example with Correlation Threshold

In [65]:
# create sample data with correlated features

# Initialize the pseudo-random number generator in the NumPy library using the integer 42 as the seed (see Hitchiker's Guide to the Galaxy)
np.random.seed(42)
df_corr = pd.DataFrame({
    'age': np.random.randint(20, 60, 100),
    'income': np.random.randint(30000, 100000, 100),
})
# birth_year is highly correlated with age
df_corr['birth_year'] = 2024 - df_corr['age'] + np.random.randint(-1, 2, 100)

# savings loosely correlated with income
df_corr['savings'] = df_corr['income'] * 0.3 + np.random.randint(-5000, 5000, 100)

corr_matrix = df_corr.corr().abs()
corr_matrix

,age,income,birth_year,savings
age,1.000000,0.020665,0.997497,0.004248
income,0.020665,1.000000,0.028015,0.910857
birth_year,0.997497,0.028015,1.000000,0.000539
savings,0.004248,0.910857,0.000539,1.000000


In [66]:
# Isolate the upper triangle of the matrix to eliminate redundancy
upper = corr_matrix.where(np.triu(np.ones((4,4)), k=1).astype('bool'))
upper

,age,income,birth_year,savings
age,NaN,0.020665,0.997497,0.004248
income,NaN,NaN,0.028015,0.910857
birth_year,NaN,NaN,NaN,0.000539
savings,NaN,NaN,NaN,NaN


In [67]:
# Define your threshold
threshold = 0.75
# indentify which features exceed correlation threshold and drop on of the pair
drop_features = [column for column in upper.columns if any(upper[column] > threshold)]
keep_columns = [column for column in upper.columns if column not in drop_features]
keep_columns

['age', 'income']

## Wrapper Methods
Use model performance to evaluate feature subsets.
>- **Forward selection**: Start empty, add features one at a time based on performance gain
>- **Backward elimination**: Start with all features, remove worst performers iteratively
>- **Recursive feature elimination (RFE)**: Train model, drop least important features, repeat

### Example with Forward Selection

In [68]:
from sklearn.datasets import make_regression

In [69]:
# Generate sample data with 10 features, only 5 are informative
X, y = make_regression(n_samples=100, n_features=10, n_informative=5, random_state=42)
df = pd.DataFrame(X, columns=[f'f{i}' for i in range(10)])
df

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9
0,1.065480,1.446978,0.196555,-0.439731,-0.186872,0.889631,-1.485560,0.082284,1.031845,0.267050
1,-0.074446,-0.342715,-0.802277,-0.420645,-1.415371,0.174578,0.404051,0.257550,-0.161286,1.886186
2,0.656554,-0.680025,0.232254,0.346448,0.250493,0.473833,-0.714351,-1.191303,0.293072,1.865775
3,1.119575,-1.515191,1.366874,-2.301921,-0.275052,0.311250,-0.249036,3.078881,1.644968,0.576557
4,-1.197878,-0.985726,0.504047,0.955142,-0.062679,-1.035242,-0.792873,-0.553649,-0.530258,-0.107030
...,...,...,...,...,...,...,...,...,...,...
95,0.514439,0.005244,0.046981,-0.599375,-1.594428,-0.142379,0.622850,0.120296,-0.450065,-1.067620
96,-0.130143,0.825416,0.813510,1.098777,-0.217681,-0.310267,0.021004,0.324166,1.305479,0.681953
97,0.243339,0.650201,-0.099176,0.872457,-1.280304,-0.691908,-1.070085,-0.045586,1.846637,-1.525525
98,-0.778305,-1.348185,0.743264,-1.710168,0.307802,0.347582,-0.183983,-0.539760,0.170865,0.018434


In [70]:
# Import SequentialFeatureSelector class from feature_selection module
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector

In [71]:
df.columns

Index(['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9'], dtype='str')

In [72]:
# Iteratively add features that improve model performance
model = LinearRegression()
sfs = SequentialFeatureSelector(model, n_features_to_select=3, direction='forward', cv=5)
sfs.fit(df, y)
# Get selected feature names
selected = df.columns[sfs.get_support()].tolist()

selected

['f1', 'f2', 'f4']

## Embedded Methods
Feature selection happens during model training.
>- **Lasso (L1 regularization)**: Shrinks weak feature coefficients to zero
>- **Ridge (L2 regularization)**: Reduces coefficient magnitude but keeps all features
>- **Tree-based importance**: Random forest / XGBoost rank features by split contribution

### Example with Lasso

In [73]:
# Import Lasso class
from sklearn.linear_model import Lasso

In [74]:
df_corr

,age,income,birth_year,savings
0,58,65222,1966,16252.6
1,48,93335,1975,26009.5
2,34,40965,1989,15095.5
3,27,54538,1997,11515.4
4,40,38110,1983,7842.0
...,...,...,...,...
95,28,77254,1996,27611.2
96,27,51918,1998,15763.4
97,31,90713,1993,24354.9
98,53,60306,1971,19708.8


In [75]:
# Lasso (Embedded)
# L1 regularization shrinks weak coefficients to exactly zero
lasso = Lasso(alpha=100)
lasso.fit(df_corr, y)

# Features with non-zero coefficients are selected
lasso_selected = df_corr.columns[lasso.coef_ != 0].tolist()
coefficients = dict(zip(df_corr.columns, np.round(lasso.coef_, 4)))

In [62]:
lasso_selected

['income', 'savings']

# The End